# 13 — Cross-Repo Integration Bridges

Demonstrates the 4 cross-repo adapters:
1. **SNN adapter** — spike trains ↔ quantum rotations
2. **SSGF adapter** — geometry matrix ↔ Hamiltonian, phase roundtrip
3. **Orchestrator mapping** — 18 quantum ↔ 35 domainpack oscillators
4. **Fusion-core disruption** — NPZ archive → ITER 11-feature vector

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

## 1. SNN Adapter

In [ ]:
from scpn_quantum_control.bridge.snn_adapter import (
    SNNQuantumBridge,
    spike_train_to_rotations,
)

rng = np.random.default_rng(42)
spikes = (rng.random((50, 4)) > 0.5).astype(float)

angles = spike_train_to_rotations(spikes, window=10)
print(f"Spike history: {spikes.shape}")
print(f"Rotation angles: {angles}")
print(f"Range: [{angles.min():.3f}, {angles.max():.3f}] (expected [0, pi])")

bridge = SNNQuantumBridge(n_neurons=2, n_inputs=4, seed=42)
out = bridge.forward(spikes)
print(f"Quantum output currents: {out}")

## 2. SSGF Adapter

In [ ]:
from qiskit.quantum_info import Statevector

from scpn_quantum_control.bridge.ssgf_adapter import (
    quantum_to_ssgf_state,
    ssgf_state_to_quantum,
    ssgf_w_to_hamiltonian,
)

W = np.array([[0, 0.3, 0.1, 0.05], [0.3, 0, 0.2, 0.1], [0.1, 0.2, 0, 0.3], [0.05, 0.1, 0.3, 0]])
omega = np.array([1.0, 1.5, 2.0, 0.8])

H = ssgf_w_to_hamiltonian(W, omega)
print(f"W: {W.shape}, Hamiltonian: {H.num_qubits} qubits, {len(H)} terms")

# Phase roundtrip
theta_in = np.array([0.5, 1.2, 2.8, -1.0])
qc = ssgf_state_to_quantum({"theta": theta_in})
sv = Statevector.from_instruction(qc)
recovered = quantum_to_ssgf_state(sv, 4)

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(4)
ax.bar(x - 0.15, theta_in, 0.3, label="Input θ", color="#1f77b4")
ax.bar(x + 0.15, recovered["theta"], 0.3, label="Recovered θ", color="#ff7f0e")
ax.set_xlabel("Oscillator")
ax.set_ylabel("Phase (rad)")
ax.set_title(f"SSGF Phase Roundtrip (R_global = {recovered['R_global']:.3f})")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 3. Orchestrator Phase Mapping

In [ ]:
from scpn_quantum_control.identity.binding_spec import (
    ARCANE_SAPIENCE_SPEC,
    ORCHESTRATOR_MAPPING,
    orchestrator_to_quantum_phases,
    quantum_to_orchestrator_phases,
)

theta_18 = np.linspace(0, 2 * np.pi, 18, endpoint=False)
orch = quantum_to_orchestrator_phases(theta_18)
theta_back = orchestrator_to_quantum_phases(orch)

diff = np.angle(np.exp(1j * (theta_back - theta_18)))
print(f"Quantum oscillators: {len(theta_18)}")
print(f"Orchestrator oscillators: {len(orch)}")
print(f"Roundtrip max error: {np.max(np.abs(diff)):.2e}")

# Layer structure
layers = ARCANE_SAPIENCE_SPEC["layers"]
print("\nLayer structure:")
for lay in layers:
    n_orch = sum(len(ORCHESTRATOR_MAPPING[oid]) for oid in lay["oscillator_ids"])
    print(f"  {lay['name']:>20}: {len(lay['oscillator_ids'])} quantum -> {n_orch} orchestrator")

## 4. Fusion-Core Disruption Adapter

In [ ]:
from scpn_quantum_control.control.q_disruption_iter import from_fusion_core_shot

# Simulate a fusion-core NPZ shot
safe_shot = {
    "Ip_MA": np.array([14.5, 14.8, 15.0]),
    "q95": np.array([3.1, 3.0, 2.9]),
    "beta_N": np.array([1.5, 1.6, 1.7]),
    "locked_mode_amp": np.array([0.0001]),
    "ne_1e19": np.array([0.85]),
    "is_disruption": 0,
}
features, label, warnings = from_fusion_core_shot(safe_shot)

print(f"Features shape: {features.shape}")
print(f"Label: {label} ({'disruption' if label else 'safe'})")
print(f"Warnings ({len(warnings)} defaulted features):")
for w in warnings:
    print(f"  {w}")